# Stage 14 V1 — сравнение xRFM с GBDT baseline

## Исследовательский вопрос

Есть ли на тех же 47 разрешённых признаках KOMUS material reserve, который раскрывается через iterative kernel / metric feature learning и supervised recursive localization xRFM, сверх сохранённого `GBDT_mean`? Это один CPU-only controlled reopen, а не model zoo.

По умолчанию notebook ничего не обучает и не предсказывает: все дорогие флаги выключены. Final test, `INN`, `Q_B1_norm` и `Q_B2_norm` не используются как predictors.

## Experiment lock и общие функции

Ниже фиксируются только контракт, CPU runtime и наблюдение за длительными операциями. Это не запускает xRFM. Неизменными остаются данные, accepted folds, comparator, 47 признаков и закрытый final test.

In [1]:
from __future__ import annotations

import hashlib
import importlib.metadata
import inspect
import json
import os
import platform
import random
import sys
import threading
import time
from contextlib import contextmanager
from pathlib import Path
from typing import Any

os.environ['OMP_NUM_THREADS'] = '8'
os.environ['MKL_NUM_THREADS'] = '8'
os.environ['OPENBLAS_NUM_THREADS'] = '8'
os.environ['NUMEXPR_NUM_THREADS'] = '8'

import numpy as np
import pandas as pd
import psutil
import torch
from IPython.display import Markdown, display
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_info, threadpool_limits
from xrfm import xRFM

ROOT = Path.cwd()
if not (ROOT / 'reports').exists():
    ROOT = ROOT.parent
DATASET = ROOT / 'data' / 'raw' / 'Data_final.xlsb'
GENERATED = ROOT / 'reports' / 'generated'
STAGE1_PATH = GENERATED / 'stage1_baseline_results_V2.json'
STAGE7_PATH = GENERATED / 'stage7_tabm_stacking_results_V1.json'
STAGE7_OOF_PATH = GENERATED / 'stage7_tabm_stacking_oof_V1.npz'
STAGE14_OOF_PATH = GENERATED / 'stage14_xrfm_oof_V1.npz'
STAGE14_RESULT_PATH = GENERATED / 'stage14_xrfm_results_V1.json'

TARGET, IDENTIFIER = 'DefMark', 'INN'
FORBIDDEN = ('Q_B1_norm', 'Q_B2_norm')
EXPECTED_DATASET_SHA = 'fc742be66d238c529daba52ccc755f774f836b7d052ed062cdf0b345080e7930'
EXPECTED_WORKING_SHA = '80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d23d9fca00091aadb45'
EXPECTED_GBDT_GINI = 0.8063993952
EXPECTED_N, EXPECTED_FINAL_N = 289614, 72404
OUTER_SEED, FOLD_SEEDS = 42, (43, 44, 45)
THREADS, PREDICT_CHUNK = 8, 4096
SMOKE_ROWS, SMOKE_QUERY_ROWS = 12288, 256
FEASIBILITY_QUERY_ROWS = 4096
SMOKE_CEILING_SECONDS = 30 * 60
FEASIBILITY_CEILING_SECONDS = 2 * 60 * 60 + 40 * 60
OOF_CEILING_SECONDS, SAFETY_FACTOR = 12 * 60 * 60, 1.5

# ОПЕРАТОР: expensive stages выключены по умолчанию.
RUN_SMOKE = False
RUN_FEASIBILITY_PREFLIGHT = False
RUN_FULL_OOF = False

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def sha256_indices(values: np.ndarray) -> str:
    return hashlib.sha256(np.ascontiguousarray(values, dtype=np.int64).tobytes()).hexdigest()

def gini(y_true: np.ndarray, score: np.ndarray) -> float:
    return float(2.0 * roc_auc_score(y_true, score) - 1.0)

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

class Stage14LiveStatus:
    """Один обновляемый русский status panel; heartbeat не вызывает API модели."""
    def __init__(self, interval_seconds: float = 30.0) -> None:
        self.interval_seconds = interval_seconds
        self.process = psutil.Process()
        self.state: dict[str, Any] = {}
        self.started_at: float | None = None
        self._handle: Any = None
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None
        self.peak_rss_bytes = 0
        self.minimum_available_ram_bytes = 2**63 - 1

    def begin(self, stage: str, fold: int) -> None:
        self.started_at = time.monotonic(); self._handle = None
        self.peak_rss_bytes = 0; self.minimum_available_ram_bytes = 2**63 - 1
        self.update(stage=stage, fold=fold, status='RUNNING', processed=0, total=0)

    def _text(self) -> str:
        now = time.monotonic()
        elapsed = 0.0 if self.started_at is None else now - self.started_at
        mem = psutil.virtual_memory()
        rss = self.process.memory_info().rss
        self.peak_rss_bytes = max(self.peak_rss_bytes, rss)
        self.minimum_available_ram_bytes = min(self.minimum_available_ram_bytes, mem.available)
        done, total = self.state.get('processed', 0), self.state.get('total', 0)
        rate = done / elapsed if elapsed > 0 and done else None
        eta = (total - done) / rate if rate and total > done else None
        worker_rss = self.state.get('worker_rss_bytes', rss); worker_peak = self.state.get('worker_peak_rss_bytes', worker_rss)
        lines = [
            f"**Stage 14 V1 · {self.state.get('status', 'RUNNING')}**",
            f"Этап: {self.state.get('stage', '—')} · Fold: {self.state.get('fold', '—')} · Подэтап: {self.state.get('substep', '—')}",
            f"Прошло: {elapsed / 60:.1f} мин · Worker RSS: {worker_rss / 2**30:.2f} GiB · peak: {worker_peak / 2**30:.2f} GiB · доступно RAM: {mem.available / 2**30:.2f} GiB",
            f"Worker RSS / RAM: {100 * worker_rss / mem.total:.1f}% · Worker PID: {self.state.get('worker_pid', '—')}",
        ]
        if total:
            lines.append(f"Прогресс: {done}/{total} ({100 * done / total:.1f}%) · rows/sec: {rate:.2f}" if rate else f"Прогресс: {done}/{total}")
        if eta is not None:
            lines.append(f"ETA: {eta / 60:.1f} мин (только по измеренной скорости)")
        return '  \n'.join(lines)

    def update(self, **kwargs: Any) -> None:
        self.state.update(kwargs)
        try:
            if self._handle is None:
                self._handle = display(Markdown(self._text()), display_id=True)
            else:
                self._handle.update(Markdown(self._text()))
        except Exception:
            print(self._text().replace('**', '').replace('  ', ' '))

    def resource_evidence(self) -> dict[str, int]:
        return {'peak_rss_bytes': int(self.peak_rss_bytes), 'minimum_available_ram_bytes': int(self.minimum_available_ram_bytes)}

    @contextmanager
    def blocking(self, stage: str, fold: int, substep: str):
        if self.started_at is None:
            self.begin(stage, fold)
        self._stop.clear()
        self.update(stage=stage, fold=fold, substep=substep, status='RUNNING')
        def heartbeat() -> None:
            while not self._stop.wait(self.interval_seconds):
                self.update()
        self._thread = threading.Thread(target=heartbeat, daemon=True)
        self._thread.start()
        try:
            yield
        except Exception:
            self.update(status='ERROR')
            raise
        finally:
            self._stop.set()
            if self._thread is not None:
                self._thread.join(timeout=1.0)

stage14_live = Stage14LiveStatus()


d:\Projects\komus-work\.venv-xrfm-v1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Environment, provenance, data и comparator guards

Эта ячейка подготовлена для ручного preflight: она доказывает CPU-only runtime, версии, dataset identity, working indices, canonical folds и реальный Gini saved comparator. До явного запуска любой stage она не выполняется.

In [2]:
XRFM_VERSION = '0.4.5'
XRFM_SOURCE_COMMIT = '0cea9ba107c0a26dc1376a4c61c0d0bfa3e0ee1e'
EXPECTED_VERSIONS = {'numpy': '2.2.5', 'scikit-learn': '1.6.1', 'tqdm': '4.67.1', 'torch': '2.8.0+cpu'}

def cpu_runtime_manifest() -> dict[str, Any]:
    assert torch.version.cuda is None, 'STOP: torch.version.cuda должен быть None'
    assert not torch.cuda.is_available(), 'STOP: CUDA недопустима для Stage 14'
    assert torch.get_num_threads() == THREADS, 'STOP: PyTorch intra-op должен быть 8'
    assert torch.get_num_interop_threads() == 1, 'STOP: PyTorch inter-op должен быть 1'
    packages = sorted(f'{dist.metadata["Name"]}=={dist.version}' for dist in importlib.metadata.distributions())
    virtual_memory = psutil.virtual_memory()
    return {
        'python': sys.version, 'xrfm': importlib.metadata.version('xrfm'), 'torch': torch.__version__,
        'torch_version_cuda': torch.version.cuda, 'numpy': np.__version__,
        'scikit_learn': importlib.metadata.version('scikit-learn'), 'tqdm': importlib.metadata.version('tqdm'),
        'os': platform.platform(), 'architecture': platform.machine(), 'cpu': platform.processor(),
        'physical_cores': psutil.cpu_count(logical=False), 'logical_cores': psutil.cpu_count(logical=True),
        'physical_ram_bytes': virtual_memory.total, 'available_ram_bytes': virtual_memory.available,
        'thread_settings': {'xrfm': THREADS, 'torch_intra_op': torch.get_num_threads(), 'torch_inter_op': torch.get_num_interop_threads()},
        'blas_threadpools': threadpool_info(), 'environment_listing_sha256': hashlib.sha256('\n'.join(packages).encode()).hexdigest(),
    }

def establish_cpu_contract() -> dict[str, Any]:
    assert sys.version_info[:3] == (3, 12, 2), 'STOP: требуется Python 3.12.2'
    assert importlib.metadata.version('xrfm') == XRFM_VERSION, 'STOP: требуется xrfm==0.4.5'
    for package, expected in EXPECTED_VERSIONS.items():
        actual = torch.__version__ if package == 'torch' else importlib.metadata.version(package)
        assert actual == expected, f'STOP: {package}={actual}, ожидается {expected}'
    assert torch.version.cuda is None and not torch.cuda.is_available(), 'STOP: требуется CPU-only PyTorch'
    torch.set_num_threads(THREADS)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError as exc:
        if torch.get_num_interop_threads() != 1:
            raise RuntimeError('STOP: невозможно установить PyTorch inter-op=1; перезапустите kernel') from exc
    global _THREADPOOL_LIMITER
    _THREADPOOL_LIMITER = threadpool_limits(limits=THREADS)
    _THREADPOOL_LIMITER.__enter__()
    assert torch.get_num_threads() == THREADS and torch.get_num_interop_threads() == 1
    return cpu_runtime_manifest()

def metric_bundle(y_true: np.ndarray, probabilities: np.ndarray) -> dict[str, float]:
    label = (probabilities >= 0.5).astype(np.int8)
    return {'Gini': gini(y_true, probabilities), 'ROC-AUC': float(roc_auc_score(y_true, probabilities)),
            'PR-AUC': float(average_precision_score(y_true, probabilities)),
            'Precision@0.5': float(precision_score(y_true, label, zero_division=0)),
            'Recall@0.5': float(recall_score(y_true, label, zero_division=0)),
            'F1@0.5': float(f1_score(y_true, label, zero_division=0))}

def load_locked_context() -> dict[str, Any]:
    assert DATASET.exists() and STAGE1_PATH.exists() and STAGE7_PATH.exists() and STAGE7_OOF_PATH.exists(), 'STOP: отсутствует required input artifact'
    assert sha256_file(DATASET) == EXPECTED_DATASET_SHA, 'STOP: SHA-256 Data_final.xlsb не совпадает'
    stage1 = json.loads(STAGE1_PATH.read_text(encoding='utf-8'))
    stage7 = json.loads(STAGE7_PATH.read_text(encoding='utf-8'))
    features = list(stage1['допустимые_признаки'])
    assert len(features) == 47 and len(set(features)) == 47
    assert not set(features).intersection(FORBIDDEN) and IDENTIFIER not in features
    assert features == stage7['raw_features_in_order'], 'STOP: accepted feature order Stage 1/7 не совпадает'
    assert stage7['dataset_sha256'] == EXPECTED_DATASET_SHA and stage7['working_index_sha256'] == EXPECTED_WORKING_SHA
    assert stage7['baseline_selection']['B_star'] == 'GBDT_mean'
    raw = pd.read_excel(DATASET, engine='pyxlsb')
    assert len(raw) == EXPECTED_N + EXPECTED_FINAL_N and TARGET in raw and IDENTIFIER in raw
    try:
        numeric = raw.loc[:, features].apply(pd.to_numeric, errors='raise')
    except Exception as exc:
        raise RuntimeError('STOP_INVALID_INPUT: accepted feature содержит nonnumeric value') from exc
    with np.load(STAGE7_OOF_PATH, allow_pickle=False) as saved:
        required = {'working_indices', 'target', 'fold', 'gbdt_mean'}
        assert required.issubset(saved.files), 'STOP: OOF artifact имеет неполный набор ключей'
        working_indices = np.asarray(saved['working_indices'], dtype=np.int64)
        y = np.asarray(saved['target'], dtype=np.int8)
        saved_fold = np.asarray(saved['fold'], dtype=np.int8)
        comparator = np.asarray(saved['gbdt_mean'], dtype=np.float64)
    assert len(working_indices) == len(y) == len(saved_fold) == len(comparator) == EXPECTED_N
    assert np.unique(working_indices).size == EXPECTED_N and sha256_indices(working_indices) == EXPECTED_WORKING_SHA
    assert np.array_equal(raw.loc[working_indices, TARGET].to_numpy(dtype=np.int8), y), 'STOP: target/index identity mismatch'
    assert np.isfinite(comparator).all() and ((0 <= comparator) & (comparator <= 1)).all()
    assert abs(gini(y, comparator) - EXPECTED_GBDT_GINI) <= 1e-9, 'STOP: saved GBDT_mean Gini не совпадает'
    splits = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=OUTER_SEED).split(np.zeros(EXPECTED_N), y))
    expected_fold = np.zeros(EXPECTED_N, dtype=np.int8)
    for fold_id, (_, valid_pos) in enumerate(splits, start=1):
        expected_fold[valid_pos] = fold_id
    assert np.array_equal(saved_fold, expected_fold), 'STOP: saved fold identity не соответствует canonical StratifiedKFold'
    assert np.setdiff1d(np.arange(len(raw)), working_indices).size == EXPECTED_FINAL_N
    X = np.ascontiguousarray(numeric.loc[working_indices].to_numpy(dtype=np.float32, copy=True))
    assert X.shape == (EXPECTED_N, 47) and np.isfinite(X).all(), 'STOP_INVALID_INPUT: NaN/Inf в accepted features'
    assert set(np.unique(y)) == {0, 1}
    return {'X': X, 'y': y, 'features': features, 'splits': splits, 'saved_fold': saved_fold, 'comparator': comparator, 'working_indices': working_indices}


## Frozen xRFM config и preprocessing

Здесь формируется ровно одна locked configuration. `rfm_params` передаётся явно. После construction проверяется effective config: `iters=3`, ключа `iterations` нет и CPU/device/thread contract не изменены. Preprocessing fit выполняется только на inner-fit.

In [3]:
RFM_PARAMS = {
    'model': {'kernel': 'l2_high_dim', 'bandwidth': 10.0, 'exponent': 1.0, 'norm_p': None,
              'bandwidth_mode': 'constant', 'agop_power': 0.5, 'diag': False, 'solver': 'solve',
              'mem_gb': 8.0, 'const_mix': 0.0, 'power': 2, 'eps': None},
    'fit': {'method': 'lstsq', 'reg': 1e-3, 'iters': 3, 'M_batch_size': 4096,
            'total_points_to_sample': 8192, 'return_best_params': True, 'early_stop_rfm': False,
            'early_stop_multiplier': 1.1, 'center_grads': False, 'prefit_eigenpro': False,
            'solver': 'solve', 'return_Ms': False, 'get_agop_best_model': False, 'verbose': False},
}
XRFM_KWARGS = {'device': 'cpu', 'n_trees': 1, 'n_tree_iters': 0, 'max_leaf_size': 8192,
               'number_of_splits': None, 'split_method': 'linear', 'classification_mode': 'zero_one',
               'tuning_metric': 'brier', 'categorical_info': None, 'fixed_vector': None, 'callback': None,
               'time_limit_s': None, 'n_threads': 8, 'refill_size': 1500, 'random_state': None,
               'split_temperature': None, 'use_temperature_tuning': False, 'overlap_fraction': 0.0,
               'keep_weight_frac_in_predict': 0.99, 'max_leaf_count_in_ensemble': 12, 'verbose': False}

def fresh_rfm_params() -> dict[str, dict[str, Any]]:
    return {'model': dict(RFM_PARAMS['model']), 'fit': dict(RFM_PARAMS['fit'])}

def construct_locked_model() -> xRFM:
    assert importlib.metadata.version('xrfm') == XRFM_VERSION, 'STOP: требуется xrfm==0.4.5'
    assert 'iterations' not in RFM_PARAMS['fit'] and RFM_PARAMS['fit']['iters'] == 3
    signature = inspect.signature(xRFM)
    for required in ('rfm_params', 'device', 'split_method', 'n_threads', 'use_temperature_tuning'):
        assert required in signature.parameters, f'STOP: xRFM API не содержит {required}'
    model = xRFM(rfm_params=fresh_rfm_params(), **XRFM_KWARGS)
    validate_resolved_config(model)
    return model

def validate_resolved_config(model: xRFM) -> None:
    assert model.rfm_params is not None, 'XRFM_CONFIG_MISMATCH: rfm_params отсутствует'
    assert model.rfm_params['fit'].get('iters') == 3, 'XRFM_CONFIG_MISMATCH: effective iters != 3'
    assert 'iterations' not in model.rfm_params['fit'], 'XRFM_CONFIG_MISMATCH: запрещённый key iterations найден'
    assert torch.device(model.device).type == 'cpu', 'XRFM_CONFIG_MISMATCH: device != cpu'
    for section in ('model', 'fit'):
        for key, value in RFM_PARAMS[section].items():
            assert model.rfm_params[section].get(key) == value, f'XRFM_CONFIG_MISMATCH: {section}.{key}'
    materialized = {'max_leaf_size': '_base_max_leaf_size', 'number_of_splits': 'number_of_splits', 'n_trees': 'n_trees',
                    'n_tree_iters': 'n_tree_iters', 'split_method': 'split_method', 'classification_mode': 'classification_mode',
                    'tuning_metric': 'tuning_metric', 'categorical_info': 'categorical_info', 'fixed_vector': 'fixed_vector',
                    'callback': 'callback', 'time_limit_s': 'time_limit_s', 'n_threads': 'n_threads',
                    'refill_size': 'min_val_size', 'split_temperature': 'split_temperature',
                    'use_temperature_tuning': 'use_temperature_tuning', 'overlap_fraction': 'overlap_fraction',
                    'keep_weight_frac_in_predict': 'keep_weight_frac_in_predict', 'max_leaf_count_in_ensemble': 'max_leaf_count_in_ensemble', 'verbose': 'verbose'}
    for key, attribute in materialized.items():
        assert getattr(model, attribute) == XRFM_KWARGS[key], f'XRFM_CONFIG_MISMATCH: {key}->{attribute}'
    constructor_only = {'device', 'random_state'}
    assert constructor_only.issubset(inspect.signature(xRFM).parameters), 'XRFM_CONFIG_MISMATCH: constructor-only argument'
    assert torch.version.cuda is None and not torch.cuda.is_available(), 'XRFM_CONFIG_MISMATCH: CUDA недопустима'

def prepare_outer_fold(context: dict[str, Any], fold_id: int, train_pos: np.ndarray, valid_pos: np.ndarray) -> dict[str, Any]:
    fold_seed = FOLD_SEEDS[fold_id - 1]
    inner_train_rel, inner_val_rel = next(StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=fold_seed).split(train_pos, context['y'][train_pos]))
    inner_train_pos, inner_val_pos = train_pos[inner_train_rel], train_pos[inner_val_rel]
    scaler = StandardScaler(with_mean=True, with_std=True)
    X_fit = np.ascontiguousarray(scaler.fit_transform(context['X'][inner_train_pos]), dtype=np.float32)
    X_inner_val = np.ascontiguousarray(scaler.transform(context['X'][inner_val_pos]), dtype=np.float32)
    X_query = np.ascontiguousarray(scaler.transform(context['X'][valid_pos]), dtype=np.float32)
    for array in (X_fit, X_inner_val, X_query):
        assert array.dtype == np.float32 and array.flags['C_CONTIGUOUS'] and np.isfinite(array).all(), 'STOP_INVALID_INPUT'
    return {'X_fit': X_fit, 'y_fit': context['y'][inner_train_pos], 'X_inner_val': X_inner_val,
            'y_inner_val': context['y'][inner_val_pos], 'X_query': X_query, 'inner_fit_rows': len(inner_train_pos),
            'inner_validation_rows': len(inner_val_pos), 'outer_validation_positions': valid_pos}

def predict_proba_chunked(model: xRFM, query: np.ndarray, fold_id: int) -> np.ndarray:
    output = np.empty((len(query), 2), dtype=np.float64)
    started = time.monotonic()
    for start in range(0, len(query), PREDICT_CHUNK):
        stop = min(start + PREDICT_CHUNK, len(query))
        probability = np.asarray(model.predict_proba(query[start:stop]), dtype=np.float64)
        assert probability.shape == (stop - start, 2) and np.isfinite(probability).all()
        assert ((0.0 <= probability) & (probability <= 1.0)).all() and np.allclose(probability.sum(axis=1), 1.0, atol=1e-6)
        output[start:stop] = probability
        stage14_live.update(processed=stop, total=len(query), fold=fold_id, substep='predict_proba')
    return output

def tree_statistics(model: xRFM) -> dict[str, Any]:
    leaves: list[dict[str, Any]] = []
    def visit(node: dict[str, Any], depth: int) -> None:
        if node['type'] == 'leaf':
            leaves.append({'depth': depth, 'train_rows': int(len(node.get('train_indices', [])))})
        else:
            visit(node['left'], depth + 1); visit(node['right'], depth + 1)
    for tree in model.trees:
        visit(tree, 0)
    rows = np.asarray([leaf['train_rows'] for leaf in leaves], dtype=np.int64)
    return {'effective_tree_depth': max((leaf['depth'] for leaf in leaves), default=0), 'leaf_count': len(leaves),
            'leaf_train_rows_min': int(rows.min()), 'leaf_train_rows_median': float(np.median(rows)), 'leaf_train_rows_max': int(rows.max())}


## Smoke, feasibility и compute gate

Следующий код подготовлен, но не вызывается при стандартных flags. Smoke использует только Fold 1 outer-train и 256 feature rows outer-validation без чтения их labels. Feasibility использует полный Fold 1 и не является quality result. Лимиты времени фиксируют operational outcome `STOPPED_BY_COMPUTE_COST`.

In [4]:
def _legacy_run_smoke_disabled(context: dict[str, Any]) -> dict[str, Any]:
    train_pos, valid_pos = context['splits'][0]
    stage14_live.begin('Smoke preflight', 1)
    subset_rel, _ = next(StratifiedShuffleSplit(n_splits=1, train_size=SMOKE_ROWS, random_state=FOLD_SEEDS[0]).split(train_pos, context['y'][train_pos]))
    subset_pos = train_pos[subset_rel]
    smoke_context = dict(context)
    smoke_context['y'] = context['y']
    prepared = prepare_outer_fold(smoke_context, 1, subset_pos, valid_pos)
    assert prepared['inner_fit_rows'] > 8192, 'STOP: smoke не достигает recursive split'
    query = prepared['X_query'][:SMOKE_QUERY_ROWS]
    seed_everything(FOLD_SEEDS[0]); model = construct_locked_model()
    fit_started = time.monotonic()
    with stage14_live.blocking('Smoke preflight', 1, 'xRFM.fit'):
        model.fit(prepared['X_fit'], prepared['y_fit'], prepared['X_inner_val'], prepared['y_inner_val'])
    fit_seconds = time.monotonic() - fit_started
    if fit_seconds > SMOKE_CEILING_SECONDS:
        return {'outcome': 'STOPPED_BY_COMPUTE_COST', 'quality': 'UNKNOWN', 'fit_seconds': fit_seconds}
    with stage14_live.blocking('Smoke preflight', 1, 'predict pass 1'):
        first = predict_proba_chunked(model, query, 1)
    with stage14_live.blocking('Smoke preflight', 1, 'predict pass 2'):
        second = predict_proba_chunked(model, query, 1)
    maximum_difference = float(np.max(np.abs(first - second)))
    assert maximum_difference <= 1e-6, 'STOP: smoke repeatability > 1e-6'
    result = {'outcome': 'PASS', 'quality': 'NOT_EVALUATED', 'subset_rows': SMOKE_ROWS, 'query_rows': SMOKE_QUERY_ROWS,
              'fit_seconds': fit_seconds, 'max_abs_diff': maximum_difference, **stage14_live.resource_evidence(),
              'tree': tree_statistics(model)}
    display(Markdown('## Smoke summary\n### FACTS\n' + json.dumps(result, ensure_ascii=False, indent=2)))
    return result

def _legacy_run_feasibility_disabled(context: dict[str, Any]) -> dict[str, Any]:
    train_pos, valid_pos = context['splits'][0]
    stage14_live.begin('Feasibility preflight', 1)
    prep_started = time.monotonic(); prepared = prepare_outer_fold(context, 1, train_pos, valid_pos)
    preprocessing_seconds = time.monotonic() - prep_started
    query = prepared['X_query'][:FEASIBILITY_QUERY_ROWS]
    seed_everything(FOLD_SEEDS[0]); model = construct_locked_model()
    fit_started = time.monotonic()
    with stage14_live.blocking('Feasibility preflight', 1, 'xRFM.fit'):
        model.fit(prepared['X_fit'], prepared['y_fit'], prepared['X_inner_val'], prepared['y_inner_val'])
    fit_seconds = time.monotonic() - fit_started
    if preprocessing_seconds + fit_seconds > FEASIBILITY_CEILING_SECONDS:
        return {'outcome': 'STOPPED_BY_COMPUTE_COST', 'quality': 'UNKNOWN', 'preprocessing_seconds': preprocessing_seconds, 'fit_seconds': fit_seconds}
    timings, passes = [], []
    for pass_id in (1, 2):
        started = time.monotonic()
        with stage14_live.blocking('Feasibility preflight', 1, f'predict pass {pass_id}'):
            passes.append(predict_proba_chunked(model, query, 1))
        timings.append(time.monotonic() - started)
    conservative_rows_per_second = min(FEASIBILITY_QUERY_ROWS / timing for timing in timings)
    result = {'outcome': 'PASS', 'quality': 'NOT_EVALUATED', 'outer_train_rows': len(train_pos),
              'inner_fit_rows': prepared['inner_fit_rows'], 'inner_validation_rows': prepared['inner_validation_rows'],
              'preprocessing_seconds': preprocessing_seconds, 'fit_seconds': fit_seconds, 'predict_seconds': timings,
              'conservative_rows_per_second': conservative_rows_per_second, 'max_abs_diff': float(np.max(np.abs(passes[0] - passes[1]))),
              **stage14_live.resource_evidence(),
              'tree': tree_statistics(model)}
    assert result['max_abs_diff'] <= 1e-6, 'STOP: feasibility repeatability > 1e-6'
    return result

def compute_gate(feasibility: dict[str, Any]) -> dict[str, Any]:
    assert feasibility['outcome'] == 'PASS'
    measured_fold_seconds = feasibility['preprocessing_seconds'] + feasibility['fit_seconds']
    projection = SAFETY_FACTOR * (3 * measured_fold_seconds + EXPECTED_N / feasibility['conservative_rows_per_second'])
    outcome = 'ELIGIBLE_FOR_MANUAL_FULL_OOF_REVIEW' if projection <= OOF_CEILING_SECONDS else 'STOPPED_BY_COMPUTE_COST'
    return {'outcome': outcome, 'measured_fold_preprocessing_plus_fit_seconds': measured_fold_seconds,
            'conservative_rows_per_second': feasibility['conservative_rows_per_second'], 'safety_factor': SAFETY_FACTOR,
            'projected_oof_seconds': projection, 'projected_oof_hours': projection / 3600, 'ceiling_hours': 12}


## Guarded full OOF

Код ниже запрещает полный OOF, пока не существуют успешные результаты smoke и feasibility и положительный compute gate. При `RUN_FULL_OOF=False` artefacts не создаются. Validation labels берутся только после предсказания соответствующего fold.

In [5]:
def _legacy_run_full_oof_disabled(context: dict[str, Any], smoke: dict[str, Any], feasibility: dict[str, Any], gate: dict[str, Any]) -> dict[str, Any]:
    assert RUN_FULL_OOF is True, 'STOP: RUN_FULL_OOF=False'
    assert smoke.get('outcome') == 'PASS', 'STOP: smoke PASS evidence отсутствует'
    assert feasibility.get('outcome') == 'PASS', 'STOP: feasibility PASS evidence отсутствует'
    assert gate.get('outcome') == 'ELIGIBLE_FOR_MANUAL_FULL_OOF_REVIEW', 'STOP: compute gate не разрешил manual review'
    full_started = time.monotonic()
    predictions = np.full(EXPECTED_N, np.nan, dtype=np.float64)
    fold_results = []
    for fold_id, (train_pos, valid_pos) in enumerate(context['splits'], start=1):
        seed_everything(FOLD_SEEDS[fold_id - 1])
        prep_started = time.monotonic(); prepared = prepare_outer_fold(context, fold_id, train_pos, valid_pos)
        preprocessing_seconds = time.monotonic() - prep_started
        model = construct_locked_model(); fit_started = time.monotonic()
        with stage14_live.blocking('Full OOF', fold_id, 'xRFM.fit'):
            model.fit(prepared['X_fit'], prepared['y_fit'], prepared['X_inner_val'], prepared['y_inner_val'])
        with stage14_live.blocking('Full OOF', fold_id, 'chunked predict_proba'):
            probability = predict_proba_chunked(model, prepared['X_query'], fold_id)[:, 1]
        predictions[valid_pos] = probability
        y_valid = context['y'][valid_pos]  # labels read only after predictions are complete.
        fold_metrics = metric_bundle(y_valid, probability)
        comparator_gini = gini(y_valid, context['comparator'][valid_pos])
        fold_results.append({'fold': fold_id, 'runtime_seconds': time.monotonic() - fit_started + preprocessing_seconds,
                             'peak_rss_bytes': psutil.Process().memory_info().rss, 'metrics': fold_metrics,
                             'comparator_gini': comparator_gini, 'delta_gini': fold_metrics['Gini'] - comparator_gini,
                             'tree': tree_statistics(model)})
    assert np.isfinite(predictions).all()
    total_metrics = metric_bundle(context['y'], predictions)
    delta = total_metrics['Gini'] - EXPECTED_GBDT_GINI
    wins = sum(item['delta_gini'] > 0 for item in fold_results); losses = sum(item['delta_gini'] < 0 for item in fold_results)
    decision = 'material_gain' if delta >= 0.010 and wins >= 2 else ('inferior' if delta <= -0.010 and losses >= 2 else 'no_material_benefit')
    fold_ginis = [item['metrics']['Gini'] for item in fold_results]
    result = {'stage': 'Stage 14 V1', 'decision': decision, 'runtime_seconds': time.monotonic() - full_started,
              'oof_metrics': total_metrics, 'delta_gini': delta, 'wins': wins, 'losses': losses,
              'fold_gini_std': float(np.std(fold_ginis)), 'folds': fold_results,
              'config': {'xrfm': XRFM_KWARGS, 'rfm_params': RFM_PARAMS}}
    np.savez_compressed(STAGE14_OOF_PATH, working_indices=context['working_indices'], target=context['y'], fold=context['saved_fold'], xrfm=predictions)
    STAGE14_RESULT_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    return result


## Запуск только по ручным flags

FACTS: в сохранённом notebook все flags равны `False`; настоящие xRFM fit/predict не выполнялись. INTERPRETATION: пока отсутствуют smoke/feasibility evidence и OOF quality. LIMITATIONS: random CV не доказывает temporal stability. NEXT STEP: после ручного preflight включить только следующую разрешённую stage.

In [6]:
print('Legacy dispatcher disabled. Используйте process-isolated Stage 14 dispatcher ниже.')


Legacy dispatcher disabled. Используйте process-isolated Stage 14 dispatcher ниже.


## Process-level watchdog и persisted manual gate

Активный path ниже запускает xRFM только в отдельном worker process. Parent heartbeat каждые 5 секунд измеряет process-tree RSS и системную RAM, может завершить worker без unsafe thread kill и сохраняет evidence только после реального успешного preflight.

In [7]:
import subprocess
import tempfile

# ОПЕРАТОР: запускаем ТОЛЬКО Smoke preflight.
RUN_SMOKE_PREFLIGHT = True
RUN_FEASIBILITY_PREFLIGHT = False
RUN_FULL_OOF = False
MANUAL_FULL_OOF_APPROVED = False
SMOKE_EVIDENCE_PATH = GENERATED / 'stage14_xrfm_smoke_V1.json'
FEASIBILITY_EVIDENCE_PATH = GENERATED / 'stage14_xrfm_feasibility_V1.json'
WORKER_PATH = ROOT / 'scripts' / 'stage14_xrfm_worker.py'
stage14_live.interval_seconds = 5.0

def hardware_guard(full_feasibility: bool = False) -> None:
    assert (psutil.cpu_count(logical=False) or 0) >= 8, 'STOP: требуется не менее 8 physical CPU cores'
    assert psutil.virtual_memory().total >= 32 * 2**30, 'STOP: требуется не менее 32 GiB physical RAM'
    if full_feasibility and psutil.virtual_memory().available < 16 * 2**30:
        raise RuntimeError('STOPPED_BY_COMPUTE_COST: available RAM < 16 GiB before feasibility')

def identity_bundle(context: dict[str, Any], manifest: dict[str, Any]) -> dict[str, Any]:
    assert WORKER_PATH.exists(), 'STOP: worker file отсутствует для provenance'
    payload = {'dataset_sha256': EXPECTED_DATASET_SHA, 'working_index_sha256': sha256_indices(context['working_indices']),
               'target_sha256': hashlib.sha256(context['y'].tobytes()).hexdigest(), 'fold_sha256': hashlib.sha256(context['saved_fold'].tobytes()).hexdigest(),
               'features_sha256': hashlib.sha256(json.dumps(context['features'], ensure_ascii=False).encode()).hexdigest(),
               'comparator_sha256': hashlib.sha256(context['comparator'].tobytes()).hexdigest(),
               'python': manifest['python'], 'xrfm': manifest['xrfm'], 'torch': manifest['torch'], 'numpy': manifest['numpy'],
               'scikit_learn': manifest['scikit_learn'], 'environment_hash': manifest['environment_listing_sha256'],
               'cpu': manifest['cpu'], 'architecture': manifest['architecture'], 'physical_cores': manifest['physical_cores'],
               'logical_cores': manifest['logical_cores'], 'physical_ram_bytes': manifest['physical_ram_bytes'], 'os': manifest['os'],
               'threads': {'xrfm': 8, 'torch_intra_op': 8, 'torch_inter_op': 1, 'blas': manifest['blas_threadpools']},
               'source_commit': XRFM_SOURCE_COMMIT, 'constructor_hash': hashlib.sha256(json.dumps(XRFM_KWARGS, sort_keys=True).encode()).hexdigest(),
               'rfm_params_hash': hashlib.sha256(json.dumps(RFM_PARAMS, sort_keys=True).encode()).hexdigest(),
               'seeds': {'outer': 42, 'folds': list(FOLD_SEEDS), 'inner': 'StratifiedShuffleSplit(test_size=0.20,random_state=fold_seed)'},
               'preprocessing': 'StandardScaler(with_mean=True,with_std=True);fit=inner-fit;float32;STOP_INVALID_INPUT',
               'worker_sha256': sha256_file(WORKER_PATH),
               'compute_policy': {'smoke_seconds': SMOKE_CEILING_SECONDS, 'feasibility_seconds': FEASIBILITY_CEILING_SECONDS, 'oof_seconds': OOF_CEILING_SECONDS, 'safety_factor': SAFETY_FACTOR, 'rss_fraction': .80, 'available_ram_bytes': 8*2**30, 'available_ram_seconds': 30, 'feasibility_ram_bytes': 16*2**30}}
    return payload

def worker_tree_rss(pid: int) -> int:
    process = psutil.Process(pid)
    return sum(item.memory_info().rss for item in [process, *process.children(recursive=True)] if item.is_running())

def isolated_worker(X_train: np.ndarray, y_train: np.ndarray, X_query: np.ndarray, fold_id: int, ceiling: float, run_mode: str) -> dict[str, Any]:
    assert run_mode in {'smoke', 'feasibility', 'full_oof'}
    config = {'seed': FOLD_SEEDS[fold_id - 1], 'run_mode': run_mode, 'rfm_params': fresh_rfm_params(), 'xrfm_kwargs': XRFM_KWARGS}
    with tempfile.TemporaryDirectory(prefix='stage14_xrfm_') as folder:
        folder = Path(folder); job, result = folder / 'job.npz', folder / 'result.json'
        np.savez_compressed(job, X_train=X_train, y_train=y_train, X_query=X_query, config=json.dumps(config))
        env = dict(os.environ, OMP_NUM_THREADS='8', MKL_NUM_THREADS='8', OPENBLAS_NUM_THREADS='8', NUMEXPR_NUM_THREADS='8')
        command = [sys.executable, str(WORKER_PATH), '--job', str(job), '--result', str(result)]
        stage14_live.begin('Stage 14 worker', fold_id); started = time.monotonic(); low_ram_since = None; peak = 0
        process = subprocess.Popen(command, env=env)
        stop_reason = None
        while process.poll() is None:
            elapsed = time.monotonic() - started; memory = psutil.virtual_memory(); rss = worker_tree_rss(process.pid); peak = max(peak, rss)
            stage14_live.update(stage='Stage 14 worker', fold=fold_id, substep='worker fit/predict', status='RUNNING', worker_pid=process.pid, worker_rss_bytes=rss, worker_peak_rss_bytes=peak)
            if elapsed >= ceiling: stop_reason = 'wall-clock ceiling'
            elif rss > .80 * memory.total: stop_reason = 'worker/process-tree RSS > 80% physical RAM'
            elif memory.available < 8 * 2**30:
                low_ram_since = low_ram_since or time.monotonic()
                if time.monotonic() - low_ram_since >= 30: stop_reason = 'available RAM < 8 GiB for >=30 seconds'
            else: low_ram_since = None
            if stop_reason:
                process.terminate()
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: process.kill(); process.wait()
                stage14_live.update(status='STOP', substep=stop_reason)
                return {'outcome': 'STOPPED_BY_COMPUTE_COST', 'quality': 'UNKNOWN', 'reason': stop_reason, 'peak_rss_bytes': peak}
            time.sleep(5)
        payload = json.loads(result.read_text(encoding='utf-8')) if result.exists() else {'status': 'ERROR', 'error': 'worker returned no evidence'}
        payload['peak_rss_bytes'] = peak
        if payload['status'] == 'MEMORY_ERROR': return {'outcome': 'STOPPED_BY_COMPUTE_COST', 'quality': 'UNKNOWN', **payload}
        if payload['status'] == 'XRFM_CONFIG_MISMATCH': return {'outcome': 'XRFM_CONFIG_MISMATCH', **payload}
        if payload['status'] != 'PASS': raise RuntimeError(payload.get('traceback', payload.get('error', 'worker ERROR')))
        with np.load(result.with_suffix('.npz'), allow_pickle=False) as output:
            payload['first'] = output['first']; payload['second'] = output['second'] if 'second' in output.files else None
        return {'outcome': 'PASS', **payload}

def run_preflight(context: dict[str, Any], manifest: dict[str, Any], feasibility: bool) -> dict[str, Any]:
    train_pos, valid_pos = context['splits'][0]
    if feasibility:
        train_for_worker, query, ceiling = train_pos, context['X'][valid_pos[:FEASIBILITY_QUERY_ROWS]], FEASIBILITY_CEILING_SECONDS
    else:
        subset, _ = next(StratifiedShuffleSplit(n_splits=1, train_size=SMOKE_ROWS, random_state=43).split(train_pos, context['y'][train_pos]))
        train_for_worker, query, ceiling = train_pos[subset], context['X'][valid_pos[:SMOKE_QUERY_ROWS]], SMOKE_CEILING_SECONDS
    result = isolated_worker(context['X'][train_for_worker], context['y'][train_for_worker], query, 1, ceiling, 'feasibility' if feasibility else 'smoke')
    if result['outcome'] == 'PASS':
        assert result['first'].shape == result['second'].shape == (len(query), 2)
        assert np.isfinite(result['first']).all() and np.allclose(result['first'].sum(1), 1, atol=1e-6)
        result['max_abs_diff'] = float(np.max(np.abs(result['first'] - result['second'])))
        result['conservative_rows_per_second'] = min(len(query) / seconds for seconds in result['predict_seconds'])
        assert result['max_abs_diff'] <= 1e-6
    result.pop('first', None); result.pop('second', None); result['identity'] = identity_bundle(context, manifest)
    result['dataset_sha256'] = EXPECTED_DATASET_SHA; result['smoke_rows'] = SMOKE_ROWS; result['query_rows'] = len(query)
    return result

def run_full_oof_isolated(context: dict[str, Any]) -> dict[str, Any]:
    oof_started_at = time.monotonic()
    predictions = np.full(EXPECTED_N, np.nan); folds = []
    for fold_id, (train_pos, valid_pos) in enumerate(context['splits'], 1):
        elapsed_oof = time.monotonic() - oof_started_at; remaining_oof_seconds = OOF_CEILING_SECONDS - elapsed_oof
        if remaining_oof_seconds <= 0: return {'outcome': 'STOPPED_BY_COMPUTE_COST', 'quality': 'UNKNOWN', 'folds': folds}
        result = isolated_worker(context['X'][train_pos], context['y'][train_pos], context['X'][valid_pos], fold_id, remaining_oof_seconds, 'full_oof')
        if result['outcome'] != 'PASS': return result
        if time.monotonic() - oof_started_at > OOF_CEILING_SECONDS: return {'outcome': 'STOPPED_BY_COMPUTE_COST', 'quality': 'UNKNOWN', 'folds': folds}
        assert result['prediction_passes'] == 1 and result['second'] is None
        probabilities = result.pop('first')[:, 1]; result.pop('second')
        predictions[valid_pos] = probabilities
        y_valid = context['y'][valid_pos]  # read only after worker returns predictions
        metrics = metric_bundle(y_valid, probabilities); comparator_gini = gini(y_valid, context['comparator'][valid_pos])
        folds.append({'fold': fold_id, 'metrics': metrics, 'comparator_gini': comparator_gini, 'delta_gini': metrics['Gini'] - comparator_gini, **result})
    overall = metric_bundle(context['y'], predictions); delta = overall['Gini'] - EXPECTED_GBDT_GINI
    wins = sum(item['delta_gini'] > 0 for item in folds); losses = sum(item['delta_gini'] < 0 for item in folds)
    decision = 'material_gain' if delta >= .010 and wins >= 2 else ('inferior' if delta <= -.010 and losses >= 2 else 'no_material_benefit')
    return {'outcome': 'PASS', 'decision': decision, 'oof_metrics': overall, 'delta_gini': delta, 'wins': wins, 'losses': losses, 'folds': folds, '_predictions': predictions}

def persisted_preflight(context: dict[str, Any], manifest: dict[str, Any]) -> dict[str, Any]:
    assert SMOKE_EVIDENCE_PATH.exists() and FEASIBILITY_EVIDENCE_PATH.exists(), 'STOP: persisted preflight evidence отсутствует'
    smoke = json.loads(SMOKE_EVIDENCE_PATH.read_text(encoding='utf-8')); feasibility = json.loads(FEASIBILITY_EVIDENCE_PATH.read_text(encoding='utf-8'))
    assert smoke.get('outcome') == 'PASS' and feasibility.get('outcome') == 'PASS'
    assert feasibility.get('compute_decision') == 'ELIGIBLE_FOR_MANUAL_FULL_OOF_REVIEW'
    current = identity_bundle(context, manifest)
    for label, evidence in (('smoke', smoke), ('feasibility', feasibility)):
        saved = evidence.get('identity', {})
        changed = next((key for key in current if saved.get(key) != current[key]), None)
        if changed: raise RuntimeError(f'PREFLIGHT_PROVENANCE_MISMATCH: {label}/{changed}')
    return feasibility

def synthetic_worker_watchdog_tests() -> None:
    with tempfile.TemporaryDirectory() as folder:
        result = Path(folder) / 'result.json'
        completed = subprocess.run([sys.executable, str(WORKER_PATH), '--synthetic', 'memory', '--result', str(result)], check=True)
        assert completed.returncode == 0 and json.loads(result.read_text())['status'] == 'MEMORY_ERROR'
        sleeping = subprocess.Popen([sys.executable, str(WORKER_PATH), '--synthetic', 'sleep', '--seconds', '60', '--result', str(folder / 'sleep.json')])
        time.sleep(.2); sleeping.terminate(); assert sleeping.wait(timeout=10) != 0
    print('Synthetic worker/watchdog tests PASS (без xRFM fit).')


In [8]:
if RUN_FULL_OOF and (RUN_SMOKE_PREFLIGHT or RUN_FEASIBILITY_PREFLIGHT):
    raise RuntimeError('STOP: full OOF требует отдельного operator run после просмотра сохранённого preflight evidence и manual approval.')
if not any((RUN_SMOKE_PREFLIGHT, RUN_FEASIBILITY_PREFLIGHT, RUN_FULL_OOF)):
    print('Stage 14 V1 safety pre-run prepared; all ML flags are False.')
elif RUN_FULL_OOF:
    assert MANUAL_FULL_OOF_APPROVED, 'STOP: MANUAL_FULL_OOF_APPROVED=False'
    manifest = establish_cpu_contract(); hardware_guard(); context = load_locked_context(); preflight = persisted_preflight(context, manifest)
    full_oof_result = run_full_oof_isolated(context)
    if full_oof_result.get('outcome') == 'PASS':
        predictions = full_oof_result.pop('_predictions')
        np.savez_compressed(STAGE14_OOF_PATH, working_indices=context['working_indices'], target=context['y'], fold=context['saved_fold'], xrfm=predictions)
        STAGE14_RESULT_PATH.write_text(json.dumps(full_oof_result, ensure_ascii=False, indent=2), encoding='utf-8')
else:
    manifest = establish_cpu_contract(); hardware_guard(RUN_FEASIBILITY_PREFLIGHT); context = load_locked_context()
    smoke = run_preflight(context, manifest, feasibility=False) if RUN_SMOKE_PREFLIGHT else json.loads(SMOKE_EVIDENCE_PATH.read_text(encoding='utf-8'))
    if RUN_SMOKE_PREFLIGHT and smoke.get('outcome') == 'PASS': SMOKE_EVIDENCE_PATH.write_text(json.dumps(smoke, ensure_ascii=False, indent=2), encoding='utf-8')
    if RUN_FEASIBILITY_PREFLIGHT:
        assert smoke.get('outcome') == 'PASS', 'STOP: feasibility требует smoke PASS'
        feasibility = run_preflight(context, manifest, feasibility=True)
        gate = compute_gate(feasibility) if feasibility.get('outcome') == 'PASS' else {'outcome': feasibility.get('outcome')}
        feasibility['smoke_outcome'] = smoke.get('outcome'); feasibility['compute_projection'] = gate
        feasibility['compute_decision'] = gate.get('outcome')
        FEASIBILITY_EVIDENCE_PATH.write_text(json.dumps(feasibility, ensure_ascii=False, indent=2), encoding='utf-8')


AssertionError: STOP: требуется не менее 8 physical CPU cores